In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)

print("========== DATA CLEANING ==========\n")

df = pd.read_csv('/Life_Expectancy_Data.csv')
print("Original Dataset:")
print(df.head(), "\n")

df.columns = df.columns.str.strip()

df.info()
print("\nDataset Description:")
print(df.describe(), "\n")

print("Missing values before cleaning:")
print(df.isnull().sum(), "\n")

df = df.drop(columns=['Population', 'GDP'])

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum(), "\n")

print(f"Shape before removing duplicates: {df.shape}")
print(f"Number of duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}\n")

df_scaled = df.copy()

for col in num_cols:
    if col == 'Year':
        continue
    s = skew(df[col])
    if -0.5 <= s <= 0.5:
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler()
    df_scaled[col] = scaler.fit_transform(df[[col]])

le = LabelEncoder()
df_scaled['Status'] = le.fit_transform(df_scaled['Status'])

corr = df_scaled.select_dtypes(include=[np.number]).corr()
life_corr = corr['Life expectancy'].abs().sort_values()
least_corr_features = life_corr.head(3).index.tolist()

least_corr_features = [col for col in least_corr_features
                       if col not in ['Life expectancy', 'Status']]

print(f"Least correlated features with Life expectancy: {least_corr_features}")
df_scaled = df_scaled.drop(columns=least_corr_features)

print("\nCleaned and scaled dataset ready!\n")

print("========== REGRESSION ==========\n")

X_reg = df_scaled.drop(columns=['Life expectancy'])
y_reg = df_scaled['Life expectancy']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_reg.shape}")
print(f"Testing set size: {X_test_reg.shape}\n")

print("Training Linear Regression model...")
lr = LinearRegression()
lr.fit(X_train_reg, y_train_reg)

y_pred_lr = lr.predict(X_test_reg)

mse_lr = mean_squared_error(y_test_reg, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test_reg, y_pred_lr)

print("Linear Regression Results:")
print(f"  MSE: {mse_lr:.4f}")
print(f"  RMSE: {rmse_lr:.4f}")
print(f"  R2 Score: {r2_lr:.4f}\n")

print("Training Random Forest Regressor...")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_reg, y_train_reg)

y_pred_rf = rf.predict(X_test_reg)

mse_rf = mean_squared_error(y_test_reg, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test_reg, y_pred_rf)

print("Random Forest Regressor Results:")
print(f"  MSE: {mse_rf:.4f}")
print(f"  RMSE: {rmse_rf:.4f}")
print(f"  R2 Score: {r2_rf:.4f}\n")

print("--- Regression Model Comparison ---")
print(f"Linear Regression R2: {r2_lr:.4f}")
print(f"Random Forest R2: {r2_rf:.4f}")
if r2_rf > r2_lr:
    print("✓ Random Forest performs better\n")
else:
    print("✓ Linear Regression performs better\n")

print("========== CLASSIFICATION ==========\n")

X_class = df_scaled.drop(columns=['Status'])
y_class = df_scaled['Status']

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class, y_class, train_size=0.7, random_state=42
)

print(f"Training set size: {X_train_class.shape}")
print(f"Testing set size: {X_test_class.shape}\n")

print("Training Logistic Regression model...")
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_class, y_train_class)

y_pred_log = log_model.predict(X_test_class)

print("Training Decision Tree Classifier...")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_class, y_train_class)

y_pred_dt = dt_model.predict(X_test_class)

print("\n--- Logistic Regression Results ---")
acc_log = accuracy_score(y_test_class, y_pred_log)
print(f"Accuracy: {acc_log:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_class, y_pred_log))
print("\nClassification Report:")
print(classification_report(y_test_class, y_pred_log))

print("\n--- Decision Tree Classifier Results ---")
acc_dt = accuracy_score(y_test_class, y_pred_dt)
print(f"Accuracy: {acc_dt:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_class, y_pred_dt))
print("\nClassification Report:")
print(classification_report(y_test_class, y_pred_dt))

print("\n--- Classification Model Comparison ---")
print(f"Logistic Regression Accuracy: {acc_log:.4f}")
print(f"Decision Tree Accuracy: {acc_dt:.4f}")
if acc_dt > acc_log:
    print("✓ Decision Tree performs better")
else:
    print("✓ Logistic Regression performs better")

========== DATA CLEANING ==========

Original Dataset:
   Year      Status  Life expectancy   Adult Mortality  infant deaths  \
0  2015  Developing              65.0            263.0             62   
1  2014  Developing              59.9            271.0             64   
2  2013  Developing              59.9            268.0             66   
3  2012  Developing              59.5            272.0             69   
4  2011  Developing              59.2            275.0             71   

   Alcohol  percentage expenditure  Hepatitis B  Measles    BMI   ...  Polio  \
0     0.01               71.279624         65.0      1154   19.1  ...    6.0   
1     0.01               73.523582         62.0       492   18.6  ...   58.0   
2     0.01               73.219243         64.0       430   18.1  ...   62.0   
3     0.01               78.184215         67.0      2787   17.6  ...   67.0   
4     0.01                7.097109         68.0      3013   17.2  ...   68.0   

   Total expenditure  Dip